In [ ]:
#Loading, Cleaning, Memory

In [1]:
from recsys_utils import *
import numpy as np, pandas as pd
mlflow = setup_mlflow()

In [2]:
# --- Load raw ---
articles     = pd.read_csv(ARTICLES_CSV)
customers    = pd.read_csv(CUSTOMERS_CSV)
transactions = pd.read_csv(transactions_path(), parse_dates=['t_dat'])

print('Loaded shapes:')
print('  articles    ', articles.shape)
print('  customers   ', customers.shape)
print('  transactions', transactions.shape)
print(f'Raw transactions memory: {mb(transactions):,.0f} MB')

Loaded shapes:
  articles     (105542, 25)
  customers    (1371980, 7)
  transactions (31788324, 5)
Raw transactions memory: 4,609 MB


In [3]:
# --- Missing values before cleaning ---
print('transactions nulls\n', transactions.isnull().sum(), '\n')
print('customers nulls\n', customers.isnull().sum(), '\n')
print('articles nulls\n', articles.isnull().sum())

transactions nulls
 t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64 

customers nulls
 customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64 

articles nulls
 article_id                        0
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_id         0
perceived_colour_value_name       0
perceived_colour_master_id        0
perceived_colour_master_name      0
department_no                     0
department_name                   0


In [4]:
# --- Clean missing values ---

customers['FN']                     = customers['FN'].fillna(0).astype('int8')
customers['Active']                 = customers['Active'].fillna(0).astype('int8')
customers['club_member_status']     = customers['club_member_status'].fillna('UNKNOWN')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')
customers['age']                    = customers['age'].fillna(customers['age'].median()).astype('float32')

articles['detail_desc']             = articles['detail_desc'].fillna('')

In [5]:

articles['article_id']     = articles['article_id'].astype('int32')
transactions['article_id'] = transactions['article_id'].astype('int32')

In [6]:
# --- Encode customer_id as a compact int32 code, using customers.csv as the authority ---
cust_categories = pd.Index(customers['customer_id'].astype(str).unique())

customers['customer_id'] = pd.Categorical(
    customers['customer_id'].astype(str), categories=cust_categories).codes.astype('int32')
transactions['customer_id'] = pd.Categorical(
    transactions['customer_id'].astype(str), categories=cust_categories).codes.astype('int32')

# Save the reverse map (code -> original string) for the final submission.
customer_map = pd.DataFrame({
    'customer_id': np.arange(len(cust_categories), dtype='int32'),
    'customer_id_original': cust_categories.values,
})
save_parquet(customer_map, PROCESSED_DIR / 'customer_map.parquet')

assert (transactions['customer_id'] >= 0).all(), 'some transaction customer was not in customers.csv'

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\customer_map.parquet  shape=(1371980, 2)


In [7]:
# --- Remaining dtypes + sort by time ---
transactions['price'] = transactions['price'].astype('float32')
transactions['sales_channel_id'] = transactions['sales_channel_id'].astype('int8')
transactions = transactions.sort_values('t_dat').reset_index(drop=True)

In [8]:

print('nulls after cleaning:')
print('  transactions', int(transactions.isnull().sum().sum()))
print('  customers   ', int(customers.isnull().sum().sum()))
print('  articles    ', int(articles.isnull().sum().sum()), '\n')
print('transactions dtypes\n', transactions.dtypes, '\n')
print(f'Transactions memory now: {mb(transactions):,.0f} MB   (was several GB as strings)')

nulls after cleaning:
  transactions 0
  customers    0
  articles     0 

transactions dtypes
 t_dat               datetime64[ns]
customer_id                  int32
article_id                   int32
price                      float32
sales_channel_id              int8
dtype: object 

Transactions memory now: 668 MB   (was several GB as strings)


In [9]:
#  Save cleaned tables
save_parquet(articles,     PROCESSED_DIR / 'articles_clean.parquet')
save_parquet(customers,    PROCESSED_DIR / 'customers_clean.parquet')
save_parquet(transactions, PROCESSED_DIR / 'transactions_clean.parquet')

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\articles_clean.parquet  shape=(105542, 25)
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\customers_clean.parquet  shape=(1371980, 7)
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\transactions_clean.parquet  shape=(31788324, 5)


WindowsPath('C:/Users/Michael Fulling/H&M Project/data/processed/transactions_clean.parquet')

In [10]:
#mlflow
with mlflow.start_run(run_name='01_data_loading_and_cleaning'):
    mlflow.log_param('articles_rows', len(articles))
    mlflow.log_param('customers_rows', len(customers))
    mlflow.log_param('transactions_rows', len(transactions))
    mlflow.log_metric('transactions_memory_mb', float(mb(transactions)))
    mlflow.log_metric('nulls_after_total', int(
        transactions.isnull().sum().sum()
        + customers.isnull().sum().sum()
        + articles.isnull().sum().sum()))
print('Logged cleaning run to MLflow.')

2026/06/11 12:06:14 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logged cleaning run to MLflow.
